# Ch 15 · Collision I: Geometry Toolkit, Pair Filtering and the Broadphase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/15-geometry-toolkit-and-broadphase.ipynb)

Companion notebook for the chapter page `chapters/15-geometry-toolkit-and-broadphase.html` ([page](../chapters/15-geometry-toolkit-and-broadphase.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## 1. The geometry toolkit
Every broadphase test is a cheap, *conservative* question: may this pair touch? Example 15.1 builds the 2-D versions of the tools in plain NumPy: AABB overlap, the separating-axis test (SAT) for two oriented boxes (4 candidate axes), and closest points between segments (the core of capsule collision). In 3-D, MJWarp's `_obb_filter` tests only the 6 face normals, not all 15 SAT axes: still conservative, cheaper.

In [ ]:
# Example 15.1 - ex1_toolkit_numpy.py
"""Example 15.1 - the geometry toolkit in plain NumPy: AABB overlap, a 2-D OBB separating-axis
test and the closest points between two segments.

These are the yes/no questions and distances that every broadphase filter and many narrowphase
colliders are built from. No MJWarp yet: only NumPy, so you can read every line.
Run: python ex1_toolkit_numpy.py
"""
import numpy as np

np.set_printoptions(precision=4, suppress=True)


def rot2(theta):
    """2-D rotation matrix; its columns are the box's local x and y axes in world coordinates."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def obb_corners(center, half, theta):
    """The 4 corners of an oriented box (center, half sizes, angle)."""
    signs = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], float)
    return center + (signs * half) @ rot2(theta).T


def aabb_of_obb(center, half, theta):
    """Axis-aligned box around an oriented box: min and max over its corners."""
    c = obb_corners(center, half, theta)
    return c.min(axis=0), c.max(axis=0)


def aabb_overlap(lo1, hi1, lo2, hi2, margin=0.0):
    """Two AABBs overlap unless they are separated along x or along y (Ericson 4.2)."""
    return bool(np.all(hi1 + margin >= lo2) and np.all(hi2 + margin >= lo1))


def obb_sat(c1, h1, t1, c2, h2, t2, margin=0.0):
    """Separating-axis test for two 2-D boxes. Candidate axes: the 2 face normals of each box.

    On each axis n, a box covers [proj - radius, proj + radius] with
    radius = sum_k |half_k * dot(axis_k, n)|. A gap on any axis proves separation.
    Returns (overlap, index of the first separating axis or -1).
    """
    axes = [rot2(t1)[:, 0], rot2(t1)[:, 1], rot2(t2)[:, 0], rot2(t2)[:, 1]]
    for k, n in enumerate(axes):
        r1 = sum(abs(h1[i] * axes[i] @ n) for i in range(2))
        r2 = sum(abs(h2[i] * axes[2 + i] @ n) for i in range(2))
        if r1 + r2 + margin < abs((c2 - c1) @ n):
            return False, k
    return True, -1


def segment_closest(p1, q1, p2, q2):
    """Closest points between segments p1-q1 and p2-q2 (Ericson 5.1.9). Returns (s, t, c1, c2, dist)."""
    d1, d2, r = q1 - p1, q2 - p2, p1 - p2
    a, e, f = d1 @ d1, d2 @ d2, d2 @ r
    c, b = d1 @ r, d1 @ d2
    denom = a * e - b * b
    s = np.clip((b * f - c * e) / denom, 0.0, 1.0) if denom > 1e-12 else 0.0  # parallel: pick s = 0
    t = (b * s + f) / e
    if t < 0.0:
        t, s = 0.0, np.clip(-c / a, 0.0, 1.0)
    elif t > 1.0:
        t, s = 1.0, np.clip((b - c) / a, 0.0, 1.0)
    c1, c2 = p1 + s * d1, p2 + t * d2
    return s, t, c1, c2, float(np.linalg.norm(c1 - c2))


# Four box configurations: box A fixed, box B moved/rotated.
A = (np.array([0.0, 0.0]), np.array([1.0, 0.5]), 0.0)
configs = {
    "far apart": (np.array([3.0, 0.0]), np.array([0.5, 0.5]), 0.0),
    "overlapping": (np.array([1.2, 0.3]), np.array([0.5, 0.5]), 0.3),
    "AABB yes, OBB no": (np.array([1.3, 0.8]), np.array([0.6, 0.15]), -np.pi / 4),
    "touching corner": (np.array([1.5, 1.0]), np.array([0.5, 0.5]), 0.0),
}
print("config             | AABB overlap | OBB (SAT) overlap | separating axis")
for name, B in configs.items():
    lo1, hi1 = aabb_of_obb(*A)
    lo2, hi2 = aabb_of_obb(*B)
    ok_aabb = aabb_overlap(lo1, hi1, lo2, hi2)
    ok_obb, axis = obb_sat(*A, *B)
    label = "-" if axis < 0 else ["A.x", "A.y", "B.x", "B.y"][axis]
    print(f"{name:18s} | {str(ok_aabb):12s} | {str(ok_obb):17s} | {label}")

print()
segs = {
    "crossing (X)": ([-1, 0], [1, 0], [0, -1], [0, 1]),
    "skew, end-to-mid": ([0, 0], [1, 0], [2, -1], [2, 1]),
    "parallel": ([0, 0], [2, 0], [1, 0.5], [3, 0.5]),
    "end to end": ([0, 0], [1, 0], [2, 1], [3, 2]),
}
for name, (p1, q1, p2, q2) in segs.items():
    s, t, c1, c2, dist = segment_closest(*(np.array(v, float) for v in (p1, q1, p2, q2)))
    print(f"{name:17s} s={s:.3f} t={t:.3f} c1={c1} c2={c2} dist={dist:.4f}")

## 2-3. The funnel and the static pre-filter
`mjw.collision` = broadphase (candidates, counter `d.ncollision`) + narrowphase (contacts, `d.nacon`). Before any of it, `put_model` builds the pair list once: all `ngeom*(ngeom-1)/2` pairs in `np.triu_indices` order, column 0 of `m.nxn_pairid` = -2 (never), -1 (default) or an explicit `<pair>` id, column 1 = collision-sensor id. Example 15.2 defines `scatter(n, seed)` (shared model M9).

In [ ]:
# Example 15.2 - ex2_scatter.py
"""Example 15.2 - the shared scatter(n, seed) scene (model M9) and its static pair list.

scatter(n) drops n free spheres and capsules (sizes 0.05-0.1) over a plane, on a 10 x 10 lattice
per layer inside a 4 x 4 m square, so the scene starts without overlap. Chapters 15 and 25 use it
for broadphase scaling. This example compiles n=20 and prints how many geom pairs exist, how many
survive the model-time pre-filter, and what m.nxn_pairid holds.
A smaller spacing (e.g. 0.22) packs the bodies so bounding volumes overlap; later examples use
that for one-shot collision calls (never stepped).
Run: python ex2_scatter.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)


def scatter(n, seed=0, spacing=0.4):
    """MJCF string: a plane plus n free bodies (alternating sphere / capsule), lattice + jitter."""
    rng = np.random.default_rng(seed)
    per_side = 10                                   # 10 x 10 bodies per layer, 4 m x 4 m
    bodies = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = -0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = -0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = 0.3 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        bodies.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return ('<mujoco model="scatter"><option timestep="0.002"/><worldbody>'
            '<light pos="0 0 4" dir="0 0 -1"/><geom name="floor" type="plane" size="3 3 .1"/>'
            + "".join(bodies) + '</worldbody></mujoco>')


mjm = mujoco.MjModel.from_xml_string(scatter(20))
m = mjw.put_model(mjm)
ngeom = mjm.ngeom
pairs = m.nxn_geom_pair_filtered.numpy()
pairid = m.nxn_pairid.numpy()                 # (all pairs, 2): [contact pair id, sensor id]

print(f"ngeom = {ngeom}, nbody = {mjm.nbody}, nv = {mjm.nv}")
print("all pairs ngeom*(ngeom-1)/2 =", ngeom * (ngeom - 1) // 2, "| m.nxn_geom_pair:", m.nxn_geom_pair.shape[0])
print("filtered pairs (NXN work list):", pairs.shape[0])
vals, counts = np.unique(pairid[:, 0], return_counts=True)
print("nxn_pairid[:, 0] values:", dict(zip(vals.tolist(), counts.tolist())))
print("nxn_pairid[:, 1] values:", dict(zip(*[v.tolist() for v in np.unique(pairid[:, 1], return_counts=True)])))
print("first 5 filtered pairs:", pairs[:5].tolist())
print("geom types:", {int(t): mujoco.mjtGeom(t).name for t in np.unique(mjm.geom_type)})
print("rbound (first 5):", m.geom_rbound.numpy()[0, :5])
print("auto broadphase:", mjw.BroadphaseType(m.opt.broadphase).name,
      "| filter:", repr(mjw.BroadphaseFilter(m.opt.broadphase_filter)))

# The scene starts with no contacts at all: no candidate survives the broadphase.
d = mjw.make_data(mjm, nworld=1, nconmax=64, njmax=256)
mjw.kinematics(m, d)
mjw.collision(m, d)
print("at t=0: ncollision =", int(d.ncollision.numpy()[0]), "| nacon =", int(d.nacon.numpy()[0]))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

# Packed variant: same bodies, spacing 0.22 m, so neighbours' bounding spheres overlap.
mjm2 = mujoco.MjModel.from_xml_string(scatter(20, spacing=0.22))
m2 = mjw.put_model(mjm2)
d2 = mjw.make_data(mjm2, nworld=1, nconmax=64, njmax=256)
mjw.kinematics(m2, d2)
mjw.collision(m2, d2)
print("packed (spacing 0.22): ncollision =", int(d2.ncollision.numpy()[0]),
      "| nacon =", int(d2.nacon.numpy()[0]))

Two geoms can collide only if `(contype1 & conaffinity2) | (contype2 & conaffinity1) != 0`. This is folded into `nxn_pairid` at model time: editing `m.geom_contype` later does nothing for rigid geoms. Re-run `put_model` instead.

In [ ]:
# Example 15.3 - ex3_contype.py
"""Example 15.3 - contype / conaffinity: which geom pairs survive the bitmask test.

Two geoms may collide only if (contype1 & conaffinity2) | (contype2 & conaffinity1) is non-zero.
put_model applies this test once, at model time, and marks failing pairs -2 in m.nxn_pairid.
Run: python ex3_contype.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

# name: (contype, conaffinity)
BITS = {"floor": (1, 1), "a": (1, 1), "b": (2, 2), "c": (2, 1), "d": (0, 0), "e": (4, 3)}

bodies = "".join(
    f'<body name="{n}" pos="{0.5 * i:.1f} 0 0.5"><freejoint/>'
    f'<geom name="{n}" type="sphere" size="0.1" contype="{ct}" conaffinity="{ca}"/></body>'
    for i, (n, (ct, ca)) in enumerate(list(BITS.items())[1:]))
XML = (f'<mujoco><worldbody><geom name="floor" type="plane" size="3 3 .1" contype="1" '
       f'conaffinity="1"/>{bodies}</worldbody></mujoco>')

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]

print("geom   contype conaffinity   (binary)")
for g, n in enumerate(names):
    ct, ca = int(mjm.geom_contype[g]), int(mjm.geom_conaffinity[g])
    print(f"{n:6s} {ct:7d} {ca:11d}   ct={ct:03b} ca={ca:03b}")

print("\nmask (ct1 & ca2) | (ct2 & ca1) for every pair; '.' = no bit in common -> pair id -2")
print("      " + " ".join(f"{n:>5s}" for n in names))
for g1, n1 in enumerate(names):
    row = []
    for g2 in range(mjm.ngeom):
        if g2 <= g1:
            row.append("     ")
            continue
        mask = ((mjm.geom_contype[g1] & mjm.geom_conaffinity[g2])
                | (mjm.geom_contype[g2] & mjm.geom_conaffinity[g1]))
        row.append(f"{int(mask):5d}" if mask else "    .")
    print(f"{n1:5s} " + " ".join(row))

pairs = m.nxn_geom_pair_filtered.numpy()
print(f"\nall pairs: {m.nxn_geom_pair.shape[0]}, kept: {pairs.shape[0]}")
print("kept pairs:", ", ".join(f"{names[i]}-{names[j]}" for i, j in pairs))
skipped = (m.nxn_pairid.numpy()[:, 0] == -2).sum()
print("pairs marked -2 in m.nxn_pairid:", int(skipped))

The other masks: same weld body, both bodies dof-less, parent-child (unless `mjDSBL_FILTERPARENT`), and `<exclude>` (signature `(body1 << 16) + body2`). The example mirrors `put_model` in NumPy and asserts it agrees. `--dump-json` writes the data of the page's NXN matrix.

In [ ]:
# Example 15.4 - ex4_exclude_parent.py
"""Example 15.4 - the other model-time filters: same body (weld), dof-less, parent-child,
<exclude>, and the FILTERPARENT disable flag.

A 3-link arm with a gripper geom on its last link, a static wall, the floor, a free box and a
mocap target (a body with no dofs of its own).
We recompute the reason each pair is dropped with NumPy (mirroring io.py::put_model) and check it
against m.nxn_pairid. Then we compile again with parent filtering disabled.
Run: python ex4_exclude_parent.py [--dump-json PATH]
  --dump-json PATH writes the geoms, bits and per-pair reasons for the page's NXN matrix.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

ARM = """
<mujoco model="arm_scene">
  <worldbody>
    <geom name="floor" type="plane" size="3 3 .1"/>
    <geom name="wall" type="box" pos="1.5 0 .5" size=".1 1 .5"/>
    <body name="link1" pos="0 0 .3">
      <joint name="j1" type="hinge" axis="0 1 0"/>
      <geom name="link1" type="capsule" fromto="0 0 0 .4 0 0" size=".05"/>
      <body name="link2" pos=".4 0 0">
        <joint name="j2" type="hinge" axis="0 1 0"/>
        <geom name="link2" type="capsule" fromto="0 0 0 .4 0 0" size=".05"/>
        <body name="link3" pos=".4 0 0">
          <joint name="j3" type="hinge" axis="0 1 0"/>
          <geom name="link3" type="capsule" fromto="0 0 0 .3 0 0" size=".05"/>
          <geom name="gripper" type="box" pos=".33 0 0" size=".03 .06 .03"/>
        </body>
      </body>
    </body>
    <body name="box" pos=".5 .6 .2"><freejoint/><geom name="box" type="box" size=".1 .1 .1"/></body>
    <body name="target" mocap="true" pos="-.5 0 .5"><geom name="target" type="sphere" size=".05"/></body>
  </worldbody>
  <contact><exclude body1="link1" body2="link3"/></contact>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def reasons(mjm):
    """Why each upper-triangle pair is dropped, mirroring io.py::put_model (first reason wins)."""
    g1, g2 = np.triu_indices(mjm.ngeom, k=1)
    b1, b2 = mjm.geom_bodyid[g1], mjm.geom_bodyid[g2]
    w1, w2 = mjm.body_weldid[b1], mjm.body_weldid[b2]
    wp1, wp2 = mjm.body_weldid[mjm.body_parentid[w1]], mjm.body_weldid[mjm.body_parentid[w2]]
    filterparent = not (mjm.opt.disableflags & mujoco.mjtDisableBit.mjDSBL_FILTERPARENT)
    mask = ((mjm.geom_contype[g1] & mjm.geom_conaffinity[g2])
            | (mjm.geom_contype[g2] & mjm.geom_conaffinity[g1])) != 0
    out = []
    for k in range(len(g1)):
        if not mask[k]:
            out.append("contype")
        elif w1[k] == w2[k]:
            out.append("same body")
        elif mjm.body_dofnum[w1[k]] == 0 and mjm.body_dofnum[w2[k]] == 0:
            out.append("no dofs")
        elif filterparent and w1[k] != 0 and w2[k] != 0 and (w1[k] == wp2[k] or w2[k] == wp1[k]):
            out.append("parent-child")
        elif ((b1[k] << 16) + b2[k]) in mjm.exclude_signature:
            out.append("exclude")
        else:
            out.append("kept")
    return g1, g2, out


def show(mjm, title):
    m = mjw.put_model(mjm)
    names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]
    g1, g2, why = reasons(mjm)
    pid = m.nxn_pairid.numpy()[:, 0]
    assert np.array_equal(pid == -2, np.array(why) != "kept"), "mirror disagrees with put_model"
    print(f"--- {title}: kept {m.nxn_geom_pair_filtered.shape[0]} of {len(g1)} pairs")
    dropped = {}
    for i, j, w in zip(g1, g2, why):
        if w != "kept":
            dropped.setdefault(w, []).append(f"{names[i]}-{names[j]}")
    for w, lst in dropped.items():
        print(f"  {w:12s}: {', '.join(lst)}")
    return m, names, g1, g2, why


mjm = mujoco.MjModel.from_xml_string(ARM)
print("geom -> body:", {mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g):
                        mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_BODY, mjm.geom_bodyid[g])
                        for g in range(mjm.ngeom)})
print("exclude_signature:", mjm.exclude_signature.tolist(), "= (body1 << 16) + body2")
m, names, g1, g2, why = show(mjm, "default")

mjm_np = mujoco.MjModel.from_xml_string(ARM)
mjm_np.opt.disableflags |= mujoco.mjtDisableBit.mjDSBL_FILTERPARENT
show(mjm_np, "FILTERPARENT disabled")

if args.dump_json:
    out = {"names": names, "ngeom": mjm.ngeom,
           "body": [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_BODY, b) for b in mjm.geom_bodyid],
           "contype": mjm.geom_contype.tolist(), "conaffinity": mjm.geom_conaffinity.tolist(),
           "pairs": [[int(i), int(j), w] for i, j, w in zip(g1, g2, why)]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

`upper_tri_index(n, i, j) = i*(2n - i - 3)//2 + j - 1` maps a pair to its row of `m.nxn_pairid`. SAP uses it to look pairs up from two geom ids.

In [ ]:
# Example 15.5 - ex5_upper_tri.py
"""Example 15.5 - upper_tri_index: how a geom pair (g1, g2) is packed into one integer.

m.nxn_pairid has one row per unordered pair g1 < g2, in np.triu_indices order. Both put_model and
the SAP sweep kernel find a pair's row with upper_tri_index(ngeom, g1, g2). We reproduce it in
Python and print the packed index of every filtered pair next to m.nxn_geom_pair_filtered.
Run: python ex5_upper_tri.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

ARM = """
<mujoco model="arm_scene">
  <worldbody>
    <geom name="floor" type="plane" size="3 3 .1"/>
    <geom name="wall" type="box" pos="1.5 0 .5" size=".1 1 .5"/>
    <body name="link1" pos="0 0 .3">
      <joint name="j1" type="hinge" axis="0 1 0"/>
      <geom name="link1" type="capsule" fromto="0 0 0 .4 0 0" size=".05"/>
      <body name="link2" pos=".4 0 0">
        <joint name="j2" type="hinge" axis="0 1 0"/>
        <geom name="link2" type="capsule" fromto="0 0 0 .4 0 0" size=".05"/>
        <body name="link3" pos=".4 0 0">
          <joint name="j3" type="hinge" axis="0 1 0"/>
          <geom name="link3" type="capsule" fromto="0 0 0 .3 0 0" size=".05"/>
          <geom name="gripper" type="box" pos=".33 0 0" size=".03 .06 .03"/>
        </body>
      </body>
    </body>
    <body name="box" pos=".5 .6 .2"><freejoint/><geom name="box" type="box" size=".1 .1 .1"/></body>
    <body name="target" mocap="true" pos="-.5 0 .5"><geom name="target" type="sphere" size=".05"/></body>
  </worldbody>
  <contact><exclude body1="link1" body2="link3"/></contact>
</mujoco>
"""



def upper_tri_index(n, i, j):
    """Row of pair (i, j), i != j, in the strict upper triangle (same formula as math.py / io.py)."""
    i, j = (j, i) if j < i else (i, j)
    return (i * (2 * n - i - 3)) // 2 + j - 1


mjm = mujoco.MjModel.from_xml_string(ARM)
m = mjw.put_model(mjm)
n = mjm.ngeom
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(n)]

# 1. The formula enumerates np.triu_indices(n, k=1) exactly.
g1, g2 = np.triu_indices(n, k=1)
assert all(upper_tri_index(n, a, b) == k for k, (a, b) in enumerate(zip(g1, g2)))
assert upper_tri_index(n, 5, 2) == upper_tri_index(n, 2, 5)          # order does not matter
print(f"ngeom = {n}: {len(g1)} packed rows, row i starts at upper_tri_index(n, i, i+1)")
print("row starts:", [upper_tri_index(n, i, i + 1) for i in range(n - 1)])

# 2. Every filtered pair, its packed index, and the nxn_pairid row it points at.
pairid = m.nxn_pairid.numpy()
print("\nfiltered pair          packed idx   nxn_pairid[idx]")
for a, b in m.nxn_geom_pair_filtered.numpy():
    idx = upper_tri_index(n, a, b)
    print(f"({a},{b}) {names[a]:>7s}-{names[b]:<7s}  {idx:10d}   {pairid[idx].tolist()}")

dropped = [int(upper_tri_index(n, a, b)) for a, b in zip(g1, g2) if pairid[upper_tri_index(n, a, b), 0] == -2]
print("\npacked indices marked -2 (never tested):", dropped)

## 4. The bounding-volume filters
`m.opt.broadphase_filter` is a bitmask of `PLANE=1 | SPHERE=2 | AABB=4 | OBB=8` (default 11 = PLANE|SPHERE|OBB), baked into the kernel with `wp.static`. Filters only change how many candidates reach the narrowphase, never the contacts.

In [ ]:
# Example 15.6 - ex6_filters.py
"""Example 15.6 - the four bounding-volume filters on one packed scene.

m.opt.broadphase_filter is a bitmask of BroadphaseFilter.PLANE | SPHERE | AABB | OBB. The filter
only decides which pairs reach the narrowphase (d.ncollision); every filter is conservative, so the
contacts the narrowphase finds (d.nacon) are the same for every choice.
Run: python ex6_filters.py [--dump-json PATH]
  --dump-json PATH writes the per-filter counts for the page's bar chart.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

F = mjw.BroadphaseFilter
ON_GPU = wp.get_device().is_cuda
NWORLD = 64 if ON_GPU else 2

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def scatter(n, seed=0, spacing=0.4):
    """MJCF string: a plane plus n free bodies (alternating sphere / capsule), lattice + jitter."""
    rng = np.random.default_rng(seed)
    per_side = 10                                   # 10 x 10 bodies per layer, 4 m x 4 m
    bodies = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = -0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = -0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = 0.3 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        bodies.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return ('<mujoco model="scatter"><option timestep="0.002"/><worldbody>'
            '<light pos="0 0 4" dir="0 0 -1"/><geom name="floor" type="plane" size="3 3 .1"/>'
            + "".join(bodies) + '</worldbody></mujoco>')



mjm = mujoco.MjModel.from_xml_string(scatter(100, spacing=0.22))   # packed: neighbours overlap
m = mjw.put_model(mjm)
nfiltered = m.nxn_geom_pair_filtered.shape[0]
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nfiltered + 64, njmax=256)  # room for every pair
mjw.kinematics(m, d)
print(f"ngeom = {mjm.ngeom}, filtered pairs per world = {nfiltered}, nworld = {NWORLD}")
print("filter                     ncollision  per world   nacon")
configs = [F.PLANE, F.SPHERE, F.PLANE | F.SPHERE, F.PLANE | F.AABB, F.PLANE | F.OBB,
           F.PLANE | F.SPHERE | F.OBB, F.PLANE | F.SPHERE | F.AABB | F.OBB]
results = []
for flt in configs:
    m.opt.broadphase_filter = flt            # baked in with wp.static: a new kernel per value
    mjw.collision(m, d)
    ncol, nacon = int(d.ncollision.numpy()[0]), int(d.nacon.numpy()[0])
    name = "|".join(f.name for f in F if f & flt)
    results.append((ncol, nacon, name))
    default = "  (default)" if flt == F.PLANE | F.SPHERE | F.OBB else ""
    print(f"{name:26s} {ncol:10d} {ncol // NWORLD:10d} {nacon:7d}{default}")
assert len({r[1] for r in results}) == 1, "nacon must not depend on the filter"
assert max(r[0] for r in results) <= d.naconmax, "candidate buffer overflowed"
print("nacon identical for every filter:", results[0][1])

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "nfiltered": nfiltered, "ngeom": mjm.ngeom,
                   "rows": [{"filter": r[2], "ncollision": r[0], "nacon": r[1]} for r in results]},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. Sweep and prune in NumPy
`sap_broadphase`: project bounding spheres on the fixed axis `normalize(0.5935, 0.7790, 0.1235)` (planes get `MJ_MAXVAL`), sort each world's lower bounds, `sap_range` (binary search for the first lower bound greater than my upper bound), `array_scan` (prefix sum over all worlds), then `nsweep = 5*nworld*ngeom` threads decode work ids into (world, i, j). The NumPy re-implementation must match MJWarp's count.

In [ ]:
# Example 15.10 - ex10_sap_numpy.py
"""Example 15.10 - sweep-and-prune re-implemented in NumPy, step by step, for two worlds.

Mirrors collision_driver.py::sap_broadphase: _sap_project (project bounding spheres onto a fixed
oblique axis), a per-world sort of the lower bounds, collision_core.py::sap_range (binary search
for how far each interval reaches), an inclusive prefix sum (wp.utils.array_scan), and the sweep
that decodes every work id into (world, i, j). The pairs that pass the PLANE | SPHERE filter must
match d.ncollision from MJWarp's SAP_TILE broadphase.
Run: python ex10_sap_numpy.py [--dump-json PATH]
  --dump-json PATH writes the intervals, sort order, ranges and prefix sum for the SAP stepper.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True, linewidth=100)
NWORLD = 2
MAXVAL = mujoco.mjMAXVAL                            # 1e10: a plane's "infinite" radius

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def scatter(n, seed=0, spacing=0.4):
    """MJCF string: a plane plus n free bodies (alternating sphere / capsule), lattice + jitter."""
    rng = np.random.default_rng(seed)
    per_side = 10                                   # 10 x 10 bodies per layer, 4 m x 4 m
    bodies = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = -0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = -0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = 0.3 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        bodies.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return ('<mujoco model="scatter"><option timestep="0.002"/><worldbody>'
            '<light pos="0 0 4" dir="0 0 -1"/><geom name="floor" type="plane" size="3 3 .1"/>'
            + "".join(bodies) + '</worldbody></mujoco>')



def upper_tri_index(n, i, j):
    i, j = (j, i) if j < i else (i, j)
    return (i * (2 * n - i - 3)) // 2 + j - 1


mjm = mujoco.MjModel.from_xml_string(scatter(12, spacing=0.22))
m = mjw.put_model(mjm)
m.opt.broadphase = mjw.BroadphaseType.SAP_TILE
m.opt.broadphase_filter = mjw.BroadphaseFilter.PLANE | mjw.BroadphaseFilter.SPHERE
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=128, njmax=64)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[1, 0::7] += np.random.default_rng(1).uniform(-0.15, 0.15, mjm.nbody - 1)  # world 1: jitter x
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)
mjw.collision(m, d)

n = mjm.ngeom
xpos = d.geom_xpos.numpy()                          # (nworld, ngeom, 3)
xmat = d.geom_xmat.numpy()
rbound = mjm.geom_rbound
margin = mjm.geom_margin + mjm.geom_gap             # no explicit <pair>: pair margin is 0
pairid = m.nxn_pairid.numpy()

# 1. _sap_project: interval [c - r, c + r] on a fixed oblique axis (a constant in the code)
direction = np.array([0.5935, 0.7790, 0.1235])
direction /= np.linalg.norm(direction)
radius = np.where(rbound == 0.0, MAXVAL, rbound) + margin
center = xpos @ direction                           # (nworld, ngeom)
lower, upper = center - radius, center + radius
print("axis =", direction.round(4), "| geom 0 is the plane: radius", radius[0])

# 2. sort the lower bounds within each world (SAP_TILE: one tile per world)
order = np.argsort(lower, axis=1, kind="stable")    # sort_index
lower_sorted = np.take_along_axis(lower, order, axis=1)

# 3. sap_range: first later sorted lower bound > my upper bound, clamped to n - 1
rng_ = np.zeros((NWORLD, n), dtype=int)
for w in range(NWORLD):
    for i in range(n):
        up = upper[w, order[w, i]]
        limit = i + 1 + np.searchsorted(lower_sorted[w, i + 1:], up, side="right")
        rng_[w, i] = min(n - 1, limit) - i

# 4. array_scan: inclusive prefix sum over ALL worlds -> total number of work packages
cumsum = np.cumsum(rng_.reshape(-1))
nwork = int(cumsum[-1])

# 5. _sap_broadphase: decode work id -> (world, i, j), skip -2 pairs, run the filter
found = []
for work in range(nwork):
    flat_i = int(np.searchsorted(cumsum, work, side="right"))     # sap_binary_search
    flat_j = flat_i + work + 1 - (cumsum[flat_i - 1] if flat_i > 0 else 0)
    w, i, j = flat_i // n, flat_i % n, flat_j % n
    g1, g2 = sorted((order[w, i], order[w, j]))
    if pairid[upper_tri_index(n, g1, g2), 0] == -2:
        continue
    if rbound[g1] == 0.0:                                         # plane band test
        ok = (xpos[w, g2] - xpos[w, g1]) @ xmat[w, g1][:, 2] <= rbound[g2] + margin[g1] + margin[g2]
    else:                                                         # bounding spheres
        bound = rbound[g1] + rbound[g2] + margin[g1] + margin[g2]
        ok = np.sum((xpos[w, g2] - xpos[w, g1]) ** 2) <= bound * bound
    if ok:
        found.append((w, int(g1), int(g2)))

for w in range(NWORLD):
    print(f"world {w}: sort_index = {order[w].tolist()}")
    print(f"         range      = {rng_[w].tolist()}")
print("cumulative_sum =", cumsum.tolist())
print(f"work packages (SAP) = {nwork}  vs  NXN threads = {NWORLD * m.nxn_geom_pair_filtered.shape[0]}")
print(f"nsweep threads launched by MJWarp = 5 * nworld * ngeom = {5 * NWORLD * n}")
print("pairs passing PLANE|SPHERE (NumPy):", len(found), "| d.ncollision (MJWarp SAP_TILE):",
      int(d.ncollision.numpy()[0]))
print("per world:", np.bincount([f[0] for f in found], minlength=NWORLD).tolist(),
      "| first pairs:", found[:4])
assert len(found) == int(d.ncollision.numpy()[0])

if args.dump_json:
    clip = lambda a: np.clip(a, -9.99, 9.99).round(3).tolist()   # planes: +-1e10 -> +-9.99
    out = {"ngeom": n, "nworld": NWORLD, "direction": direction.round(4).tolist(),
           "type": mjm.geom_type.tolist(), "rbound": rbound.round(3).tolist(),
           "xpos": xpos.astype(np.float64).round(3).tolist(), "center": center.round(3).tolist(),
           "lower": clip(lower), "upper": clip(upper), "order": order.tolist(),
           "range": rng_.tolist(), "cumsum": cumsum.tolist(), "found": found}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 7. Choosing and switching the broadphase
`put_model` picks NXN below 250,000 listed pairs, else SAP_TILE below 1000 geoms, else SAP_SEGMENTED. `m.opt.broadphase` can be switched between calls (new kernels are compiled the first time). `override_model` is a private API (`mujoco_warp._src.io`).

In [ ]:
# Example 15.7 - ex7_switch_broadphase.py
"""Example 15.7 - switching the broadphase: NXN, SAP_TILE and SAP_SEGMENTED.

m.opt.broadphase is a plain Python field read when collision() runs, so you can switch it at any
time (each value builds its own kernels the first time). With the default filter (which includes
SPHERE) all three find the same candidate pairs and the same contacts; only the work they do to get
there differs. With PLANE alone, SAP's 1-D interval test is tighter than NXN's filter, so the
candidate counts differ (the contacts do not).
Run: python ex7_switch_broadphase.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.io import override_model   # private API: may change

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 4
NSTEP = 200 if ON_GPU else 60


def scatter(n, seed=0, spacing=0.4):
    """MJCF string: a plane plus n free bodies (alternating sphere / capsule), lattice + jitter."""
    rng = np.random.default_rng(seed)
    per_side = 10                                   # 10 x 10 bodies per layer, 4 m x 4 m
    bodies = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = -0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = -0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = 0.3 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        bodies.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return ('<mujoco model="scatter"><option timestep="0.002"/><worldbody>'
            '<light pos="0 0 4" dir="0 0 -1"/><geom name="floor" type="plane" size="3 3 .1"/>'
            + "".join(bodies) + '</worldbody></mujoco>')



# 1. One-shot collision on the packed scene.
mjm = mujoco.MjModel.from_xml_string(scatter(100, spacing=0.22))
m = mjw.put_model(mjm)
print("auto choice:", mjw.BroadphaseType(m.opt.broadphase).name,
      f"({m.nxn_geom_pair_filtered.shape[0]} filtered pairs < 250000)")
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=512, njmax=1024)
mjw.kinematics(m, d)
print("broadphase      ncollision  nacon")
for bp in mjw.BroadphaseType:
    m.opt.broadphase = bp
    mjw.collision(m, d)
    print(f"{bp.name:14s} {int(d.ncollision.numpy()[0]):11d} {int(d.nacon.numpy()[0]):6d}")

# 1b. With PLANE alone, NXN keeps every non-plane pair but SAP never visits pairs whose intervals
#     are apart, so ncollision differs. The narrowphase rejects the extras: nacon is the same.
mjm30 = mujoco.MjModel.from_xml_string(scatter(30, spacing=0.22))
m30 = mjw.put_model(mjm30)
d30 = mjw.make_data(mjm30, nworld=1, nconmax=600, njmax=64)
mjw.kinematics(m30, d30)
m30.opt.broadphase_filter = mjw.BroadphaseFilter.PLANE
print("PLANE only, packed scatter(30), nworld=1:")
for bp in mjw.BroadphaseType:
    m30.opt.broadphase = bp
    mjw.collision(m30, d30)
    print(f"{bp.name:14s} {int(d30.ncollision.numpy()[0]):11d} {int(d30.nacon.numpy()[0]):6d}")

# 2. The same switch as a string override (what the CLI tools use).
override_model(m, ["opt.broadphase=SAP_SEGMENTED", "opt.broadphase_filter=PLANE|SPHERE"])
print("after override:", mjw.BroadphaseType(m.opt.broadphase).name,
      repr(mjw.BroadphaseFilter(m.opt.broadphase_filter)))

# 3. Step the unpacked scene with each broadphase and compare the trajectories.
mjm = mujoco.MjModel.from_xml_string(scatter(30))
mjm.opt.timestep = 0.005                  # bigger step so the bodies land within NSTEP steps
final = {}
for bp in mjw.BroadphaseType:
    m = mjw.put_model(mjm)
    m.opt.broadphase = bp
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=64, njmax=256)
    for _ in range(NSTEP):
        mjw.step(m, d)
    final[bp.name] = d.qpos.numpy()
    print(f"{bp.name:14s} after {NSTEP} steps: nacon = {int(d.nacon.numpy()[0])}, "
          f"overflow: {'none' if not d.overflow.numpy().any() else d.overflow.numpy()}")
ref = final["NXN"]
for name, q in final.items():
    print(f"max |qpos - qpos_NXN| for {name:14s} = {np.abs(q - ref).max():.3g}")

Benchmark: warm up once per configuration, `wp.synchronize()` before reading the clock, take the median. On a GPU, raise `SIZES` and `NWORLD` and look for the crossover (stretch exercise).

In [ ]:
# Example 15.8 - ex8_scaling.py
"""Example 15.8 - how collision time scales with the number of geoms, for each broadphase.

Times mjw.collision (broadphase + narrowphase) on packed scatter(n) scenes. Warm up once per
configuration (JIT compile), synchronize before reading the clock, report the median of a few
calls. CPU numbers say nothing about GPU performance: run this on Colab for real numbers.
Run: python ex8_scaling.py [--dump-json PATH] [--plot PATH]
"""
import argparse
import json
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

ON_GPU = wp.get_device().is_cuda
NWORLD = 64 if ON_GPU else 2
# GPU sizes stop at 3000: from n = 4000 MuJoCo's default arena is too small and compiling scatter(n)
# fails with "Could not allocate memory" (add <size memory="1G"/> to the MJCF to go higher).
SIZES = [50, 200, 800, 2000, 3000] if ON_GPU else [50, 200, 800, 2000]
REPEAT = 20 if ON_GPU else 3

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--plot", default=None)
args, _ = parser.parse_known_args()


def scatter(n, seed=0, spacing=0.4):
    """MJCF string: a plane plus n free bodies (alternating sphere / capsule), lattice + jitter."""
    rng = np.random.default_rng(seed)
    per_side = 10                                   # 10 x 10 bodies per layer, 4 m x 4 m
    bodies = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = -0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = -0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = 0.3 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        bodies.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return ('<mujoco model="scatter"><option timestep="0.002"/><worldbody>'
            '<light pos="0 0 4" dir="0 0 -1"/><geom name="floor" type="plane" size="3 3 .1"/>'
            + "".join(bodies) + '</worldbody></mujoco>')



def time_collision(m, d, repeat):
    mjw.collision(m, d)                       # warm-up: compiles kernels for this configuration
    wp.synchronize()
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        mjw.collision(m, d)
        wp.synchronize()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


rows = []
print(f"device: {wp.get_device()} | nworld = {NWORLD} | median of {REPEAT} calls")
print("     n   pairs/world  auto           NXN [ms]  SAP_TILE [ms]  SAP_SEG [ms]  ncollision")
for n in SIZES:
    mjm = mujoco.MjModel.from_xml_string(scatter(n, spacing=0.22))
    m = mjw.put_model(mjm)
    auto = mjw.BroadphaseType(m.opt.broadphase).name
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8 * n, njmax=64)
    mjw.kinematics(m, d)
    ms, ncol = {}, set()
    for bp in mjw.BroadphaseType:
        m.opt.broadphase = bp
        ms[bp.name] = 1e3 * time_collision(m, d, REPEAT)
        ncol.add(int(d.ncollision.numpy()[0]))
    assert len(ncol) == 1                     # every broadphase finds the same candidates
    nfiltered = m.nxn_geom_pair_filtered.shape[0]
    rows.append({"n": n, "pairs": nfiltered, "auto": auto, **ms, "ncollision": ncol.pop()})
    r = rows[-1]
    print(f"{n:6d} {nfiltered:12d}  {auto:13s} {r['NXN']:9.2f} {r['SAP_TILE']:14.2f} "
          f"{r['SAP_SEGMENTED']:13.2f} {r['ncollision']:11d}")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "nworld": NWORLD, "repeat": REPEAT,
                   "rows": [{k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()}
                            for r in rows]}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

if args.plot:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("skipped: matplotlib not installed")
    else:
        ns = [r["n"] for r in rows]
        for bp in mjw.BroadphaseType:
            plt.loglog(ns, [r[bp.name] for r in rows], "o-", label=bp.name)
        plt.xlabel("geoms per world (n)")
        plt.ylabel("mjw.collision time [ms]")
        plt.title(f"{wp.get_device()}, nworld={NWORLD}")
        plt.legend()
        plt.savefig(args.plot, dpi=100)
        print("saved", args.plot)

In [ ]:
# Plot the timings of Example 15.8 (uses `rows` from the cell above). CPU timings say nothing
# about GPU performance: on a Colab GPU this plot shows YOUR crossover between NXN and SAP.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("skipped: matplotlib not installed")
else:
    ns = [r["n"] for r in rows]
    for name in ["NXN", "SAP_TILE", "SAP_SEGMENTED"]:
        plt.loglog(ns, [r[name] for r in rows], "o-", label=name)
    plt.xlabel("geoms per world (n)")
    plt.ylabel("mjw.collision time [ms]")
    plt.title(f"{wp.get_device()}, nworld={NWORLD}")
    plt.legend()
    plt.show()

## 10. Lab: a broadphase-like kernel
One thread per (world, filtered pair), like `_nxn_broadphase`: bounding-sphere test plus the plane band, counted with `wp.atomic_add`. The counts must equal `d.ncollision` under SPHERE and PLANE|SPHERE.

In [ ]:
# Example 15.9 - ex9_sphere_filter_kernel.py
"""Example 15.9 - write a broadphase-like filter kernel yourself.

One thread per (world, filtered pair), like _nxn_broadphase: read the two geom centres from
d.geom_xpos, the bounding radii from m.geom_rbound and margin + gap from the model, test the
bounding spheres (and, optionally, the plane band) and count passing pairs with wp.atomic_add.
The counts must equal d.ncollision under BroadphaseFilter.SPHERE and PLANE | SPHERE.
Run: python ex9_sphere_filter_kernel.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

F = mjw.BroadphaseFilter
ON_GPU = wp.get_device().is_cuda
NWORLD = 256 if ON_GPU else 4


def scatter(n, seed=0, spacing=0.4):
    """MJCF string: a plane plus n free bodies (alternating sphere / capsule), lattice + jitter."""
    rng = np.random.default_rng(seed)
    per_side = 10                                   # 10 x 10 bodies per layer, 4 m x 4 m
    bodies = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = -0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = -0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = 0.3 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        bodies.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return ('<mujoco model="scatter"><option timestep="0.002"/><worldbody>'
            '<light pos="0 0 4" dir="0 0 -1"/><geom name="floor" type="plane" size="3 3 .1"/>'
            + "".join(bodies) + '</worldbody></mujoco>')



@wp.kernel
def sphere_filter_count(
    # Model:
    nxn_geom_pair: wp.array[wp.vec2i],
    nxn_pairid: wp.array[wp.vec2i],
    geom_rbound: wp.array2d[float],
    geom_margin: wp.array2d[float],
    geom_gap: wp.array2d[float],
    # Data in:
    geom_xpos_in: wp.array2d[wp.vec3],
    geom_xmat_in: wp.array2d[wp.mat33],
    # In:
    use_plane: bool,
    # Out:
    count_out: wp.array[int],
    per_world_out: wp.array[int],
):
    worldid, pairid = wp.tid()
    pair = nxn_geom_pair[pairid]
    g1, g2 = pair[0], pair[1]
    # batched model fields: row worldid % nbatch (shape[0] is 1 unless randomized per world)
    r1 = geom_rbound[worldid % geom_rbound.shape[0], g1]
    r2 = geom_rbound[worldid % geom_rbound.shape[0], g2]
    m1 = geom_margin[worldid % geom_margin.shape[0], g1] + geom_gap[worldid % geom_gap.shape[0], g1]
    m2 = geom_margin[worldid % geom_margin.shape[0], g2] + geom_gap[worldid % geom_gap.shape[0], g2]
    p1 = geom_xpos_in[worldid, g1]
    p2 = geom_xpos_in[worldid, g2]

    passed = bool(False)
    if r1 == 0.0 or r2 == 0.0:               # rbound == 0 marks a plane
        passed = True                        # no PLANE bit: planes always pass
        if use_plane:
            if r1 == 0.0:
                n = wp.vec3(geom_xmat_in[worldid, g1][0, 2], geom_xmat_in[worldid, g1][1, 2],
                            geom_xmat_in[worldid, g1][2, 2])
                passed = wp.dot(p2 - p1, n) <= r2 + m1 + m2
            else:
                n = wp.vec3(geom_xmat_in[worldid, g2][0, 2], geom_xmat_in[worldid, g2][1, 2],
                            geom_xmat_in[worldid, g2][2, 2])
                passed = wp.dot(p1 - p2, n) <= r1 + m1 + m2
    else:
        bound = r1 + r2 + m1 + m2
        dif = p2 - p1
        passed = wp.dot(dif, dif) <= bound * bound
    if passed or nxn_pairid[pairid][1] >= 0:   # collision-sensor pairs skip the filter
        wp.atomic_add(count_out, 0, 1)
        wp.atomic_add(per_world_out, worldid, 1)


mjm = mujoco.MjModel.from_xml_string(scatter(60, spacing=0.22))
mjm.geom_margin[1:] = 0.01                     # a small margin on every body geom
m = mjw.put_model(mjm)
nfiltered = m.nxn_geom_pair_filtered.shape[0]
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nfiltered + 16, njmax=64)

# lower the bodies by a different amount in each world, so the plane band test differs per world
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 2::7] = np.linspace(0.3, 0.05, NWORLD)[:, None]          # z of every free body
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)

print(f"ngeom = {mjm.ngeom}, filtered pairs = {nfiltered}, nworld = {NWORLD}, threads = {NWORLD * nfiltered}")
for flt, use_plane in [(F.SPHERE, False), (F.PLANE | F.SPHERE, True)]:
    count = wp.zeros(1, dtype=int)
    per_world = wp.zeros(NWORLD, dtype=int)
    wp.launch(sphere_filter_count, dim=(NWORLD, nfiltered),
              inputs=[m.nxn_geom_pair_filtered, m.nxn_pairid_filtered, m.geom_rbound,
                      m.geom_margin, m.geom_gap, d.geom_xpos, d.geom_xmat, use_plane],
              outputs=[count, per_world])
    m.opt.broadphase_filter = flt
    mjw.collision(m, d)
    mine, theirs = int(count.numpy()[0]), int(d.ncollision.numpy()[0])
    print(f"{flt!r:32s} my kernel: {mine:5d}   d.ncollision: {theirs:5d}   "
          f"match: {mine == theirs}")
    print("   my per-world counts (first 4):", per_world.numpy()[:4].tolist())
    assert mine == theirs
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## Explicit pairs
A `<contact><pair>` overrides every static rule (here contype 0) and brings its own margin, gap, condim and friction.

In [ ]:
# Example 15.11 - ex11_explicit_pair.py
"""Example 15.11 - explicit <contact><pair>: a pair id >= 0 in m.nxn_pairid and its own material.

The ball has contype=conaffinity=0, so the bitmask test alone would drop ball-floor (-2). The
explicit <pair> overrides that: put_model writes the pair's index into column 0 of nxn_pairid,
the broadphase filter uses the pair's margin + gap, and the narrowphase takes friction, condim,
solref and margin from the pair instead of mixing the two geoms' values.
Run: python ex11_explicit_pair.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

XML = """
<mujoco model="explicit_pair">
  <worldbody>
    <geom name="floor" type="plane" size="3 3 .1" friction="1 0.005 0.0001"/>
    <body name="ball" pos="0 0 0.105"><freejoint/>
      <geom name="ball" type="sphere" size="0.1" contype="0" conaffinity="0"/></body>
    <body name="box" pos="0.5 0 0.099"><freejoint/>
      <geom name="box" type="box" size="0.1 0.1 0.1"/></body>
  </worldbody>
  <contact>
    <pair geom1="ball" geom2="floor" condim="4" friction="0.3 0.3 0.01 0.0001 0.0001" margin="0.01"/>
  </contact>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]
g1, g2 = np.triu_indices(mjm.ngeom, k=1)
print("npair =", mjm.npair, "| pair_geom1/2 =", mjm.pair_geom1.tolist(), mjm.pair_geom2.tolist())
for k, (a, b) in enumerate(zip(g1, g2)):
    print(f"row {k}: {names[a]:5s}-{names[b]:5s} nxn_pairid = {m.nxn_pairid.numpy()[k].tolist()}")
print("filtered work list:", m.nxn_geom_pair_filtered.numpy().tolist())

d = mjw.make_data(mjm, nworld=2, nconmax=16, njmax=64)
mjw.kinematics(m, d)
mjw.collision(m, d)
n = min(int(d.nacon.numpy()[0]), d.naconmax)
geom = d.contact.geom.numpy()[:n]
print(f"\nnacon = {n} (ball sits 0.005 above the floor: inside the pair margin 0.01)")
for c in range(n):
    if d.contact.worldid.numpy()[c] != 0:
        continue
    print(f"world 0 {names[geom[c, 0]]}-{names[geom[c, 1]]:5s} dist={d.contact.dist.numpy()[c]:+.4f} "
          f"dim={d.contact.dim.numpy()[c]} includemargin={d.contact.includemargin.numpy()[c]:.3f} "
          f"friction={d.contact.friction.numpy()[c]}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## Exercise 15.1 (auto-checked)
Choose `contype`/`conaffinity` for the three links (same for all) so that only the three floor pairs survive, without `<exclude>`.

In [ ]:
# Exercise 15.1: a robot that touches the floor but not itself (no <exclude> allowed)
import mujoco
import numpy as np
import mujoco_warp as mjw

LINK_BITS = 'contype="1" conaffinity="1"'      # <- your answer (these defaults keep l1-l3: change them)
ROBOT = f"""
<mujoco>
  <worldbody>
    <geom name="floor" type="plane" size="3 3 .1"/>
    <body name="l1" pos="0 0 .3"><joint type="hinge" axis="0 1 0"/>
      <geom name="l1" type="capsule" fromto="0 0 0 .4 0 0" size=".05" {LINK_BITS}/>
      <body name="l2" pos=".4 0 0"><joint type="hinge" axis="0 1 0"/>
        <geom name="l2" type="capsule" fromto="0 0 0 .4 0 0" size=".05" {LINK_BITS}/>
        <body name="l3" pos=".4 0 0"><joint type="hinge" axis="0 1 0"/>
          <geom name="l3" type="capsule" fromto="0 0 0 .4 0 0" size=".05" {LINK_BITS}/>
        </body>
      </body>
    </body>
  </worldbody>
</mujoco>"""
mjm = mujoco.MjModel.from_xml_string(ROBOT)
m = mjw.put_model(mjm)
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]
kept = {(names[i], names[j]) for i, j in m.nxn_geom_pair_filtered.numpy()}
try:
    assert kept == {("floor", "l1"), ("floor", "l2"), ("floor", "l3")}, kept
    print("ok")
except AssertionError:
    print("not solved yet: kept pairs are", sorted(kept))

In [ ]:
#@title Exercise 15.1: solution (double-click to show the code)
# Exercise 15.1: a robot that touches the floor but not itself (no <exclude> allowed)
import mujoco
import numpy as np
import mujoco_warp as mjw

LINK_BITS = 'contype="2" conaffinity="1"'      # links: (2&1)|(2&1)=0; floor: (2&1)|(1&1)=1
ROBOT = f"""
<mujoco>
  <worldbody>
    <geom name="floor" type="plane" size="3 3 .1"/>
    <body name="l1" pos="0 0 .3"><joint type="hinge" axis="0 1 0"/>
      <geom name="l1" type="capsule" fromto="0 0 0 .4 0 0" size=".05" {LINK_BITS}/>
      <body name="l2" pos=".4 0 0"><joint type="hinge" axis="0 1 0"/>
        <geom name="l2" type="capsule" fromto="0 0 0 .4 0 0" size=".05" {LINK_BITS}/>
        <body name="l3" pos=".4 0 0"><joint type="hinge" axis="0 1 0"/>
          <geom name="l3" type="capsule" fromto="0 0 0 .4 0 0" size=".05" {LINK_BITS}/>
        </body>
      </body>
    </body>
  </worldbody>
</mujoco>"""
mjm = mujoco.MjModel.from_xml_string(ROBOT)
m = mjw.put_model(mjm)
names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) for g in range(mjm.ngeom)]
kept = {(names[i], names[j]) for i, j in m.nxn_geom_pair_filtered.numpy()}
assert kept == {("floor", "l1"), ("floor", "l2"), ("floor", "l3")}, kept
print("ok")

## Exercise 15.2 (auto-checked)
Write the 3-D OBB test with the 6 face axes only and check it against `_obb_filter` (filter = OBB only, so `d.ncollision == 1` exactly when it passes) on 20 random poses.

In [ ]:
# Exercise 15.2: your 6-axis OBB test against _obb_filter
TWO_BOXES = """
<mujoco><worldbody>
  <body name="a"><freejoint/><geom type="box" size=".3 .1 .05"/></body>
  <body name="b"><freejoint/><geom type="box" size=".2 .2 .02"/></body>
</worldbody></mujoco>"""


def quat_to_mat(q):            # MuJoCo (w, x, y, z)
    w, x, y, z = q
    return np.array([[1-2*(y*y+z*z), 2*(x*y-z*w), 2*(x*z+y*w)],
                     [2*(x*y+z*w), 1-2*(x*x+z*z), 2*(y*z-x*w)],
                     [2*(x*z-y*w), 2*(y*z+x*w), 1-2*(x*x+y*y)]])

def obb_sat_3d(c1, R1, h1, c2, R2, h2, margin=0.0):
    ...                        # your code: True if no face axis separates the boxes

if obb_sat_3d(np.zeros(3), np.eye(3), np.ones(3), np.zeros(3), np.eye(3), np.ones(3)) is None:
    print("not solved yet: obb_sat_3d returns None")
else:
    try:
        mjm = mujoco.MjModel.from_xml_string(TWO_BOXES)
        m = mjw.put_model(mjm)
        m.opt.broadphase_filter = mjw.BroadphaseFilter.OBB
        d = mjw.make_data(mjm, nworld=1, nconmax=8, njmax=32)
        rng = np.random.default_rng(0)
        agree = npass = 0
        h1, h2 = mjm.geom_size[0], mjm.geom_size[1]
        for k in range(20):
            q = np.zeros(14); q[3] = 1.0
            q[7:10] = rng.uniform(-0.45, 0.45, 3)
            qb = rng.normal(size=4); q[10:14] = qb / np.linalg.norm(qb)
            d.qpos.assign(q[None].astype(np.float32))
            mjw.kinematics(m, d); mjw.collision(m, d)
            mj = int(d.ncollision.numpy()[0]) == 1
            mine = obb_sat_3d(q[0:3], np.eye(3), h1, q[7:10], quat_to_mat(q[10:14]), h2)
            agree += mine == mj; npass += mj
        print(agree, npass)
        assert agree == 20
        print("Exercise 15.2 passed")
    except (AssertionError, TypeError) as e:
        print("not solved yet: fewer than 20 poses agree with _obb_filter", e)

In [ ]:
#@title Exercise 15.2: solution (double-click to show the code)
# Exercise 15.2 solution (expected output: 20 6)
TWO_BOXES = """
<mujoco><worldbody>
  <body name="a"><freejoint/><geom type="box" size=".3 .1 .05"/></body>
  <body name="b"><freejoint/><geom type="box" size=".2 .2 .02"/></body>
</worldbody></mujoco>"""


def quat_to_mat(q):            # MuJoCo (w, x, y, z)
    w, x, y, z = q
    return np.array([[1-2*(y*y+z*z), 2*(x*y-z*w), 2*(x*z+y*w)],
                     [2*(x*y+z*w), 1-2*(x*x+z*z), 2*(y*z-x*w)],
                     [2*(x*z-y*w), 2*(y*z+x*w), 1-2*(x*x+y*y)]])

def obb_sat_3d(c1, R1, h1, c2, R2, h2, margin=0.0):
    """6 face axes only, like _obb_filter (no edge-edge cross axes)."""
    for n in list(R1.T) + list(R2.T):
        r1 = np.sum(np.abs(h1 * (R1.T @ n)))
        r2 = np.sum(np.abs(h2 * (R2.T @ n)))
        if r1 + r2 + margin < abs((c2 - c1) @ n):
            return False
    return True

mjm = mujoco.MjModel.from_xml_string(TWO_BOXES)
m = mjw.put_model(mjm)
m.opt.broadphase_filter = mjw.BroadphaseFilter.OBB
d = mjw.make_data(mjm, nworld=1, nconmax=8, njmax=32)
rng = np.random.default_rng(0)
agree = npass = 0
h1, h2 = mjm.geom_size[0], mjm.geom_size[1]
for k in range(20):
    q = np.zeros(14); q[3] = 1.0
    q[7:10] = rng.uniform(-0.45, 0.45, 3)
    qb = rng.normal(size=4); q[10:14] = qb / np.linalg.norm(qb)
    d.qpos.assign(q[None].astype(np.float32))
    mjw.kinematics(m, d); mjw.collision(m, d)
    mj = int(d.ncollision.numpy()[0]) == 1
    mine = obb_sat_3d(q[0:3], np.eye(3), h1, q[7:10], quat_to_mat(q[10:14]), h2)
    agree += mine == mj; npass += mj
print(agree, npass)
assert agree == 20

## Exercise 15.3 (auto-checked)
Write `pair_of(n, k)`, the inverse of `upper_tri_index`, and check it on every row and against `m.nxn_geom_pair`.

In [ ]:
# Exercise 15.3: invert upper_tri_index
def upper_tri_index(n, i, j):
    i, j = (j, i) if j < i else (i, j)
    return (i * (2 * n - i - 3)) // 2 + j - 1

def pair_of(n, k):
    ...                                          # your code

if pair_of(2, 0) is None:
    print("not solved yet: pair_of returns None")
else:
    try:
        for n in (2, 5, 8, 50):
            for k in range(n * (n - 1) // 2):
                i, j = pair_of(n, k)
                assert 0 <= i < j < n and upper_tri_index(n, i, j) == k
        mjm = mujoco.MjModel.from_xml_string(
            '<mujoco><worldbody><geom type="plane" size="1 1 .1"/>'
            + "".join(f'<body pos="{k} 0 1"><freejoint/><geom size=".1"/></body>' for k in range(6))
            + '</worldbody></mujoco>')
        m = mjw.put_model(mjm)
        pairs = m.nxn_geom_pair.numpy()
        assert all(tuple(pairs[k]) == pair_of(mjm.ngeom, k) for k in range(len(pairs)))
        print("ok:", len(pairs), "rows checked")
    except (AssertionError, TypeError) as e:
        print("not solved yet:", type(e).__name__, e)

In [ ]:
#@title Exercise 15.3: solution (double-click to show the code)
# Exercise 15.3 solution (expected output: ok: 21 rows checked)
def upper_tri_index(n, i, j):
    i, j = (j, i) if j < i else (i, j)
    return (i * (2 * n - i - 3)) // 2 + j - 1

def pair_of(n, k):
    """Inverse of upper_tri_index: packed row k -> (i, j) with i < j."""
    i = 0
    while upper_tri_index(n, i, n - 1) < k:     # last entry of row i
        i += 1
    return i, k - upper_tri_index(n, i, i + 1) + i + 1

for n in (2, 5, 8, 50):
    for k in range(n * (n - 1) // 2):
        i, j = pair_of(n, k)
        assert 0 <= i < j < n and upper_tri_index(n, i, j) == k
mjm = mujoco.MjModel.from_xml_string(
    '<mujoco><worldbody><geom type="plane" size="1 1 .1"/>'
    + "".join(f'<body pos="{k} 0 1"><freejoint/><geom size=".1"/></body>' for k in range(6))
    + '</worldbody></mujoco>')
m = mjw.put_model(mjm)
pairs = m.nxn_geom_pair.numpy()
assert all(tuple(pairs[k]) == pair_of(mjm.ngeom, k) for k in range(len(pairs)))
print("ok:", len(pairs), "rows checked")

## Stretch: where does SAP beat NXN on your GPU?
Run Example 15.8 with sizes 100, 200, 400, ..., 6400 and `NWORLD` 256 and 4096. From n = 4000 up, add `<size memory="1G"/>` to the `scatter()` MJCF (MuJoCo's default arena is too small and compiling fails with 'Could not allocate memory'), and expect `put_model` to take tens of seconds. Find the `n` where SAP_TILE first beats NXN and compare it with the auto rule (250,000 pairs, about 708 geoms). Then place all bodies on a line perpendicular to the SAP direction (`p @ (0.5935, 0.7790, 0.1235) = const`) and explain what happens to the number of work packages.